# rl19_suite

> Phase 19 (`specs/multiagent/phase19/`): the evaluation cells and their runners. `contestSuite` is FR-C1's grid: 16 agents in the chosen squads at liars 0, 2, 4, 6 and 8 of 16 and both lifetimes; the warm start on the plain board in the same squads (the damage reference) and with `enforceK` 1-3; held out: 8 and 32 agents, mistaken members and colluding liars; arm 4 and the warm start over sequences of 6 boards at 8, 16 and 32 agents (at 16 also with the most active quarter of contesters removed after board 3 and back after board 5); the best arm in 64-agent squads. `runSequences` and `evalSequences` play the sequences with per-agent records; `warmCells` and `warmTable` measure `WARM_ANSWERS` in the speed check; `pilotCells` and `pilotGain` the squad-size pilot's selection.

In [ ]:
#| default_exp rl19_suite

In [ ]:
#| hide
from nbdev.showdoc import *

## Cells

In [ ]:
#| export
import json
import time
from pathlib import Path

import numpy as np

from explRL.bench_grid import readCSV, rowKey, writeCSV
from explRL.clue_grid import BOARD, CLUE_SETTINGS, clueCell
from explRL.clue_roll import coopRows, rollout
from explRL.colony_agents import scriptedFaults
from explRL.contest import Records
from explRL.contest_eval import contestsOf
from explRL.llm_runs import LLMConfig, cellRow
from explRL.rl19_grid import BASE19, LIFETIMES, PILOT19_PLAN, RL19_PLAN, SPEED19, VARIED, needWorld19
from explRL.rl19_step import SEQ

GATE_LIARS, EVAL19_SEEDS, HELD_SIZES, SEQ_SIZES, K19, FAULTS19 = (0, 2, 4, 6, 8), list(range(900, 910)), (8, 32), (8, 16, 32), (1, 2, 3), ('error', 'collude')


def isOpen(model):
    "Arm 4's adapters (no assignment: evaluated without guard slots)."
    return model.startswith('a19open')


def world19e(S, board='colony', **kw):
    "An evaluation world: Phase 11 settings, two holders per clue, squads of S, the Phase 19 columns (p19); 'colony': the counting board."
    return {**{k: CLUE_SETTINGS[k] for k in BOARD}, 'r': 2, 'squads': int(S), 'p19': True, **({'colony': True} if board == 'colony' else {}), **kw}


def _cell(m, test, N, sd, w, nLiar=0, turns=40, **extra):
    return dict(clueCell('contestEval', f'{m}|{test}', N, sd, 'local', channel='blackboard', model=f'local:{m}', memory='notes',
                         world=w, turns=turns, nLiar=nLiar), **extra)


def _guards(m, N):
    return {} if isOpen(m) else dict(guards=N // 4)


def seqCells(m, S, seeds=EVAL19_SEEDS, records=True):
    """Sequences of 6 boards (board p of sequence sd: seed 10 sd + p) at 8, 16 and 32 agents with N // 4 liars in arm 4's world
    (contest credit open to all, no guard slots), and at 16 also 'seqrm16' (the removal); records as P19-5 decided."""
    w = world19e(S, scheme='contest', contestSlots='all')
    return [_cell(m, f'{kind}{N}', N, 10 * sd + p, w, N // 4, seq=dict(id=sd, pos=p, remove=kind == 'seqrm', records=bool(records)))
            for N in SEQ_SIZES for kind in ('seq',) + (('seqrm',) if N == 16 else ()) for sd in seeds for p in range(SEQ)]


def contestSuite(models, squads, best=(), warm='warm', seeds=EVAL19_SEEDS, records=True, world=None):
    """FR-C1 per model (levels '<model>|<test>'): the gate grid 'gL<liars>T<life>' (16 agents, guards 4 except arm 4's); held
    out 'n<N>L<liars>' (N 8 and 32, clean and N // 4 liars), the faults 'error' and 'collude' (4 of 16); for the warm start also
    the plain board ('plain' + the test: the damage reference) and enforceK ('k<K>L<liars>T<life>'); for arm 4 and the warm
    start the sequences (seqCells); for the models in `best`, 64-agent squads joined by bridges ('big64L0', 'big64L16'; the warm
    start's plain board too). 40 turns, seeds 900-909, the default reward scheme. `world` (RL19_PLAN's): refused by needWorld19
    until bridged rollouts exist, and so are the 64-agent cells (always bridged)."""
    needWorld19(RL19_PLAN['world'] if world is None else world)
    if best:
        needWorld19('bridges')                # the 64-agent squads are joined by bridges whatever the training world
    S, cells = int(squads), []
    for m in [warm] + [x for x in models if x != warm]:
        tests = [(f'gL{nl}T{L}', 16, nl, dict(life=L)) for nl in GATE_LIARS for L in LIFETIMES]
        tests += [(f'n{N}L{nl}', N, nl, {}) for N in HELD_SIZES for nl in (0, N // 4)] + [(f, 16, 4, dict(fault=f)) for f in FAULTS19]
        for test, N, nl, kw in tests:
            cells += [_cell(m, test, N, sd, world19e(S, **_guards(m, N), **kw), nl) for sd in seeds]
            if m == warm:
                cells += [_cell(m, 'plain' + test, N, sd, world19e(S, 'plain', **kw), nl) for sd in seeds]
            if m == warm and test.startswith('g'):
                cells += [_cell(m, f'k{K}' + test[1:], N, sd, world19e(S, **_guards(m, N), enforceK=K, **kw), nl) for K in K19 for sd in seeds]
        if isOpen(m) or m == warm:
            cells += seqCells(m, S, seeds, records)
    for m in best:
        for nl in (0, 16):
            cells += [_cell(m, f'big64L{nl}', 64, sd, world19e(S, **_guards(m, 64)), nl) for sd in seeds]
            cells += [_cell(warm, f'plainbig64L{nl}', 64, sd, world19e(S, 'plain'), nl) for sd in seeds] if warm not in best else []
    return cells


def warmCells(squads, guards=4, seeds=SPEED19['warmSeeds']):
    """The speed check's WARM_ANSWERS cells ('trainG<guards>L<liars>T<life>'): the warm start on every training board type, 16
    agents, training turns, in arm 2's world (guards 4) or arm 4's (guards 0: contest credit open to all), SPEED19's world."""
    needWorld19(SPEED19['world'])
    w = dict(BASE19['world'], squads=int(squads), scheme='contest', **(dict(guards=guards, contestSlots='guards') if guards else dict(contestSlots='all')))
    return [_cell('warm', f'trainG{guards}L{nl}T{L}', 16, sd, world19e(squads, **{**w, 'life': L}), nl, BASE19['turns'])
            for nl in VARIED[0] for L in LIFETIMES for sd in seeds]


def warmTable(rows, guards=4):
    "WARM_ANSWERS for one squad size and guard count: [((16, liars, lifetime), mean honest answers per board)] from warmCells' rows."
    by = {}
    for r in rows:
        if f'|trainG{guards}L' in r['level']:
            by.setdefault((int(r['N']), int(r['nLiar']), int(float(r['world.life']))), []).append(float(r['honestAnswers']))
    return [(k, float(np.mean(v))) for k, v in sorted(by.items())]


def pilotCells(models, squads, seeds=PILOT19_PLAN['heldOut']):
    "The pilot's held-out boards: 16 agents at 4 liars, guards 4, the chosen squads, lifetime 30, 40 turns ('<model>|pilotL4')."
    return [_cell(m, 'pilotL4', 16, sd, world19e(squads, guards=4), 4) for m in models for sd in seeds]


def pilotGain(rows, model, warm='warm'):
    "The selection rule's gain: mean honest welfare (welfareHonest) of the model minus the warm start's on the pilot boards."
    v = lambda m: float(np.mean([float(r['welfareHonest']) for r in rows if r['level'] == f'{m}|pilotL4']))
    return v(model) - v(warm)

## Sequences

In [ ]:
#| export
class SubRecords:
    "A Records seen through a slot map (a board with agents removed): slot i is agent slotMap[i]."
    def __init__(self, rec, slotMap):
        self.rec, self.map = rec, list(slotMap)

    def line(self, i):
        return self.rec.line(self.map[i])

    def addBoard(self, board, slots):
        net, log = board.netByAgent(), getattr(board, 'contestLog', None)
        won = log.earned(board.nAgents) if log is not None else [0.0] * board.nAgents
        for i in slots:
            self.rec.add(self.map[i], 'answers', net[i])
            self.rec.add(self.map[i], 'contests', won[i])
        self.rec.endBoard()


def topContesters(counts, nLiar, n):
    "The n honest agents with the most contests (ties: the lower slot)."
    return sorted(range(nLiar, len(counts)), key=lambda i: (-counts[i], i))[:n]


def runSequences(cells, policyFor, logDir=None):
    """Play seqCells' sequences, every sequence of a level in one batch per board position, each with its own Records (per
    agent, when the cells ask for records). On a removal level the N // 4 agents with the most contests on boards 1-3 sit
    out boards 4-5 (the board has N - N // 4 slots, the same number of squads) and return for board 6. Rows: cellRow plus
    seqId, pos and agents (JSON: the original agent of each slot)."""
    rows, levels = [], {}
    for c in cells:
        levels.setdefault(c['level'], []).append(c)
    for cs in levels.values():
        base, sq = LLMConfig.fromDict(cs[0]['cfg']), cs[0]['seq']
        N, nl, seqs = base.N, base.nLiar, sorted({c['seq']['id'] for c in cs})
        at = {(c['seq']['id'], c['seq']['pos']): c for c in cs}
        recs = {sd: Records(N, SEQ) for sd in seqs} if sq['records'] else None
        counts, maps = {sd: np.zeros(N) for sd in seqs}, {sd: list(range(N)) for sd in seqs}
        policies, memory = policyFor(cs[0])
        for p in range(SEQ):
            drop = sq['remove'] and p in (3, 4)
            for sd in seqs:
                out = topContesters(counts[sd], nl, N // 4) if drop else []
                maps[sd] = [i for i in range(N) if i not in out]
            cfg = base
            if drop:
                n2 = N - N // 4
                cfg = LLMConfig.fromDict({**cs[0]['cfg'], 'N': n2, 'world': {**base.world, 'squads': n2 // (N // base.world['squads'])}})
            rl = [recs[sd] if not drop else SubRecords(recs[sd], maps[sd]) for sd in seqs] if recs else None
            ro = rollout(cfg, [at[(sd, p)]['seed'] for sd in seqs], policies, scriptedFaults(cfg.world, nl), memory, records=rl)
            for b, (sd, r) in enumerate(zip(seqs, coopRows(cfg, ro))):
                if p < 3:
                    for c in contestsOf(ro.boards[b], nl):
                        counts[sd][maps[sd][c[1]]] += 1
                row = dict(cellRow(at[(sd, p)], cfg, r), seqId=sd, pos=p, agents=json.dumps(maps[sd]))
                rows.append(row)
                if logDir:
                    f = Path(logDir) / (rowKey(row).replace('|', '__') + '.jsonl')
                    f.parent.mkdir(parents=True, exist_ok=True)
                    recsb = sorted((x for lg in ro.logs[b] for x in lg), key=lambda x: (x['t'], x['agent']))
                    f.write_text(''.join(json.dumps(dict(key=rowKey(row), episode=0, **x), sort_keys=True) + '\n' for x in recsb))
    return sorted(rows, key=rowKey)


def evalSequences(cells, models, out, cache=None, logDir=None):
    "As clue_roll.evalModels for sequence cells: per model out/seq_<name>.csv, read back when present (a rerun continues)."
    from explRL.local_policy import LocalPolicy
    from explRL.rl_torch import freeGPU
    out, rows = Path(out), []
    out.mkdir(parents=True, exist_ok=True)
    for name, (path, adapter) in models.items():
        mine, f = [c for c in cells if c['level'].startswith(name + '|') and 'seq' in c], out / f'seq_{name}.csv'
        if f.exists() or not mine:
            rows += readCSV(f) if f.exists() else []
            continue
        engine = LocalPolicy(path, cache=cache)
        pol = engine if adapter is None else LocalPolicy(path, adapter=adapter, cache=cache, share=engine, tokenizer=engine.tok())
        t0 = time.time()
        got = runSequences(mine, lambda c: (pol, 'notes'), logDir)
        writeCSV(got, f)
        rows += got
        print(f'{name}: {len(got)} sequence boards, {time.time() - t0:.0f} s', flush=True)
        freeGPU(*{id(p): p for p in (pol, engine)}.values())
    return sorted(rows, key=rowKey)

## Tests

In [ ]:
# FR-C1: the grid's cells, the plain-board reference and enforceK for the warm start only, arm 4 without guard slots, the
# sequences for arm 4 and the warm start, the 64-agent cells for the best arm; every cell builds a valid configuration
from collections import Counter
from explRL.rl19_suite import *
from explRL.clue_colony import checkOptions
import explRL.rl19_grid as G
for bad in (lambda: contestSuite(['warm'], 4), lambda: warmCells(4), lambda: contestSuite(['warm'], 4, best=['warm'], world='squads')):
    try:                                       # bridges (P19-1, the 64-agent cells) need Phase 20's rollouts
        bad()
        raise AssertionError('bridges accepted')
    except NotImplementedError as e:
        assert 'Phase 20' in str(e)
C = contestSuite(['warm', 'a19contest_s1', 'a19open_s1'], 4, world='squads')
lv = Counter(c['level'] for c in C)
assert lv['a19contest_s1|gL4T15'] == 10 and lv['warm|plaingL4T15'] == 10 and lv['warm|k3L8T30'] == 10 and 'a19contest_s1|k1L0T15' not in lv
assert {t.split('|')[1] for t in lv if t.startswith('a19contest_s1|')} == {f'gL{n}T{L}' for n in GATE_LIARS for L in (15, 30)} | \
    {'n8L0', 'n8L2', 'n32L0', 'n32L8', 'error', 'collude'}
assert lv['a19open_s1|seqrm16'] == 60 and lv['warm|seq32'] == 60 and not any(t.startswith('a19contest_s1|seq') for t in lv)
assert all(c['cfg']['world'].get('guards') is None for c in C if c['level'].startswith('a19open'))
assert all(c['cfg']['world']['guards'] == c['N'] // 4 for c in C if c['level'].startswith('a19contest') and 'plain' not in c['level'])
assert all('colony' not in c['cfg']['world'] and 'guards' not in c['cfg']['world'] for c in C if '|plain' in c['level'])
assert {c['cfg']['world']['fault'] for c in C if c['level'].endswith(('error', 'collude'))} == {'error', 'collude'}
assert all(c['cfg']['world']['life'] == 15 for c in C if c['level'].endswith('T15')) and all(c['cfg']['turns'] == 40 for c in C)
for c in C:
    checkOptions(c['cfg']['world'], c['N'], c['cfg']['nLiar'])
G.SPEED19['world'] = 'squads'                     # the fallback world (Gate C1 failed), for the offline tests only
try:
    W4, W0 = warmCells(4), warmCells(8, guards=0)
finally:
    G.SPEED19['world'] = G.DECISIONS19['P19_1']['value']
assert len(W4) == 30 and {(c['cfg']['nLiar'], c['cfg']['world']['life']) for c in W0} == {(n, L) for n in (0, 4, 6) for L in (15, 30)}
assert all(c['cfg']['turns'] == 20 and c['cfg']['world']['scheme'] == 'contest' for c in W4 + W0)
assert all(c['cfg']['world']['guards'] == 4 and c['cfg']['world']['contestSlots'] == 'guards' for c in W4)
assert all('guards' not in c['cfg']['world'] and c['cfg']['world']['contestSlots'] == 'all' for c in W0) and W0[0]['level'].startswith('warm|trainG0L')
assert topContesters([0, 0, 5, 1, 5, 0, 2, 0], 2, 2) == [2, 4] and topContesters([9, 0, 0, 1], 1, 1) == [3]

In [ ]:
# the runners offline (scripted replies): WARM_ANSWERS from the warm cells, the pilot gain, and the sequences with records and the
# removal (boards 4-5 without the most active quarter of contesters, the same number of squads; everyone back on board 6)
from explRL.clue_policy import claudePolicy
from explRL.clue_roll import runEval
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
pol = claudePolicy(LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
short = lambda cs: [dict(c, cfg=dict(c['cfg'], turns=3)) for c in cs]
import explRL.rl19_grid as G
G.SPEED19['world'] = 'squads'
try:
    cells = [c for g in (4, 0) for c in warmCells(4, g) if c['seed'] == 960]
finally:
    G.SPEED19['world'] = G.DECISIONS19['P19_1']['value']
rows = runEval(short(cells), lambda c: (pol, 'notes'))
for g in (4, 0):
    tab = warmTable(rows, g)
    assert [k for k, _ in tab] == [(16, n, L) for n in (0, 4, 6) for L in (15, 30)] and all(v >= 0 for _, v in tab)
pr = runEval(short(pilotCells(['warm', 'x'], 4, [950, 951])), lambda c: (pol, 'notes'))
assert pilotGain(pr, 'x') == 0.0                          # the same replies: no gain
seq = short([c for c in seqCells('warm', 4, [900, 901]) if c['level'] in ('warm|seqrm16', 'warm|seq8')])
sr = runSequences(seq, lambda c: (pol, 'notes'))
assert all(k in sr[0] for k in ('waitAnswers', 'usedAnswers', 'misinformedFirst', 'honestTurns', 'contestLags', 'agentSubmits', 'agentOpens'))
assert len(sr) == 24 and sorted({r['level'] for r in sr}) == ['warm|seq8', 'warm|seqrm16']
rm = sorted((r for r in sr if r['level'] == 'warm|seqrm16'), key=lambda r: (r['seqId'], r['pos']))
assert [int(r['N']) for r in rm[:6]] == [16, 16, 16, 12, 12, 16] and [int(r['world.squads']) for r in rm[:6]] == [4, 4, 4, 3, 3, 4]
assert all(len(json.loads(r['agents'])) == int(r['N']) for r in rm) and json.loads(rm[5]['agents']) == list(range(16))
assert json.loads(rm[3]['agents']) == json.loads(rm[4]['agents']) and all(a in json.loads(rm[3]['agents']) for a in range(4))   # liars stay

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()